In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import yfinance as yf
from scipy.stats import norm
from datetime import date
import warnings
warnings.filterwarnings("ignore")

print("All imports OK")

All imports OK


In [2]:
# Cell 3 — fetch ticker info and available expirations
qqq = yf.Ticker("QQQ")
spot = qqq.fast_info["last_price"]
expirations = qqq.options

print(f"Spot price: ${spot:.2f}")
print(f"Available expirations ({len(expirations)}):")
for e in expirations[:10]:
    print(f"  {e}")

Spot price: $729.03
Available expirations (31):
  2026-05-26
  2026-05-27
  2026-05-28
  2026-05-29
  2026-06-01
  2026-06-02
  2026-06-03
  2026-06-04
  2026-06-05
  2026-06-12


In [3]:
# Cell 4 — pull the nearest expiration chain and inspect it
exp = expirations[0]
chain = qqq.option_chain(exp)

calls = chain.calls
puts  = chain.puts

print(f"Expiration: {exp}")
print(f"Calls: {len(calls)} rows   Puts: {len(puts)} rows")
print(f"\nColumns: {list(calls.columns)}")

Expiration: 2026-05-26
Calls: 109 rows   Puts: 109 rows

Columns: ['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask', 'change', 'percentChange', 'volume', 'openInterest', 'impliedVolatility', 'inTheMoney', 'contractSize', 'currency']


In [4]:
# Cell 5 — look at the actual data
print("=== CALLS (first 5) ===")
calls[["strike","openInterest","impliedVolatility","volume","bid","ask"]].head()

# print("\n=== PUTS (first 5) ===")
# print(puts[["strike","openInterest","impliedVolatility","volume","bid","ask"]].head())

=== CALLS (first 5) ===


,strike,openInterest,impliedVolatility,volume,bid,ask
0,505.0,8,3.253908,24.0,221.11,224.6
1,525.0,29,2.947268,29.0,201.20,204.6
2,570.0,1,2.291020,1.0,156.20,159.6
3,575.0,3,2.220708,NaN,151.20,154.6
4,580.0,1,2.150395,NaN,146.20,149.6


In [5]:
# Cell 6 — check data quality
print("=== CALLS quality check ===")
print(f"  OI = 0 or NaN:  {(calls['openInterest'].fillna(0) == 0).sum()} / {len(calls)} rows")
print(f"  IV = 0 or NaN:  {(calls['impliedVolatility'].fillna(0) == 0).sum()} / {len(calls)} rows")

print("\n=== PUTS quality check ===")
print(f"  OI = 0 or NaN:  {(puts['openInterest'].fillna(0) == 0).sum()} / {len(puts)} rows")
print(f"  IV = 0 or NaN:  {(puts['impliedVolatility'].fillna(0) == 0).sum()} / {len(puts)} rows")

=== CALLS quality check ===
  OI = 0 or NaN:  0 / 109 rows
  IV = 0 or NaN:  0 / 109 rows

=== PUTS quality check ===
  OI = 0 or NaN:  5 / 109 rows
  IV = 0 or NaN:  0 / 109 rows


## compute gamma for each strike

> Black–Scholes Equation

> Call Option Price

$$
C = S_0 N(d_1) - K e^{-rt} N(d_2)
$$

Where:

$$
d_1 = \frac{\ln\left(\frac{S_0}{K}\right) + \left(r + \frac{\sigma^2}{2}\right)t}{\sigma \sqrt{t}}
$$

$$
d_2 = d_1 - \sigma \sqrt{t}
$$

In [6]:
# Cell 7 — Black-Scholes gamma function
def bs_gamma(S, K, T, r, sigma):
    """Gamma is identical for calls and puts."""
    if T <= 0 or sigma <= 0:
        return 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return norm.pdf(d1) / (S * sigma * np.sqrt(T))

print("=== bs_gamma defined OK ===")

=== bs_gamma defined OK ===


In [7]:
# Cell 8 — compute time to expiry
from datetime import date

today   = date.today()
exp_date = date.fromisoformat(exp)
T = max((exp_date - today).days / 365.0, 1/365)

print(f"Expiry: {exp}")
print(f"Days to expiry: {(exp_date - today).days}")
print(f"T (years): {T:.4f}")

Expiry: 2026-05-26
Days to expiry: 0
T (years): 0.0027


> Gamma Exposure (GEX) Formula

$$
GEX = \Gamma \times OI \times ContractSize \times S^2 \times 0.01
$$

Where:

- `Γ` = option gamma  
- `OI` = open interest  
- `ContractSize` = usually 100 shares  
- `S` = underlying stock price  
- `0.01` = converts exposure to a 1% move basis

In [8]:
# Cell 9 — compute GEX for calls and puts, combine into one DataFrame
RATE = 0.05  # risk-free rate

def compute_gex_df(df, spot, T, sign):
    rows = []
    for _, row in df.iterrows():
        K  = float(row["strike"])
        oi = float(row["openInterest"])
        iv = float(row["impliedVolatility"])
        if oi <= 0 or iv <= 0:
            continue
        g   = bs_gamma(spot, K, T, RATE, iv)
        # gex = sign * g * oi * 100 * spot**2 * 0.01
        gex = sign * g * oi * 100
        rows.append({"strike": K, "oi": oi, "iv": iv, "gamma": g, "gex": gex})
    return pd.DataFrame(rows)

calls_gex = compute_gex_df(calls, spot, T, +1)
puts_gex  = compute_gex_df(puts,  spot, T, -1)

print(f"Calls GEX rows: {len(calls_gex)}")
print(f"Puts  GEX rows: {len(puts_gex)}")
print("\nCalls sample:")
calls_gex.head(3)

Calls GEX rows: 109
Puts  GEX rows: 104

Calls sample:


,strike,oi,iv,gamma,gex
0,505.0,8.0,3.253908,0.000260,0.208353
1,525.0,29.0,2.947268,0.000311,0.902215
2,570.0,1.0,2.291020,0.000489,0.048935


In [9]:
# Cell 10 — aggregate net GEX per strike (calls + puts combined)
combined = pd.concat([calls_gex, puts_gex])
gex_by_strike = combined.groupby("strike")["gex"].sum().reset_index()
gex_by_strike = gex_by_strike.sort_values("strike").reset_index(drop=True)

print(f"Total strikes: {len(gex_by_strike)}")
print(f"Net GEX total: {gex_by_strike['gex'].sum():,.0f}")
gex_by_strike.head(10)

Total strikes: 122
Net GEX total: 163,749


,strike,gex
0,495.0,-0.086740
1,505.0,0.208353
2,515.0,-0.031742
3,520.0,-1.311074
4,525.0,-2.885303
5,530.0,-0.055170
6,535.0,-0.551262
7,540.0,-0.536992
8,545.0,-0.705346
9,550.0,-2.733292


In [10]:
# Cell 11 — formatting helper
def fmt(n):
    a = abs(n)
    p = "+" if n > 0 else ""
    if a >= 1e9: return f"{p}{n/1e9:.2f}B"
    if a >= 1e6: return f"{p}{n/1e6:.1f}M"
    if a >= 1e3: return f"{p}{n/1e3:.1f}K"
    return f"{p}{n:.0f}"

print("fmt defined OK")

fmt defined OK


## Update

In [ ]:
# Cell 12 — parameters you control
NUM_EXPIRIES  = 1    # how many expirations to aggregate
NUM_STRIKES   = 50   # strikes to show centered around spot (try 50, 75, 100)
RATE          = 0.05

# specific expirations — edit dates here
EXPIRIES_LIST = ["2026-05-30"]

In [12]:
# Cell 13 — aggregate GEX across multiple expirations
today = date.today()
all_gex = []

for exp in expirations[:NUM_EXPIRIES]:
    try:
        chain    = qqq.option_chain(exp)
        exp_date = date.fromisoformat(exp)
        T        = max((exp_date - today).days / 365.0, 1/365)

        for df, sign in [(chain.calls, +1), (chain.puts, -1)]:
            df = df.copy()
            df = df[df["openInterest"].fillna(0) > 0]
            df = df[df["impliedVolatility"].fillna(0) > 0]
            df["T"]      = T
            df["sign"]   = sign
            df["expiry"] = exp
            all_gex.append(df)
    except Exception as e:
        print(f"Skipping {exp}: {e}")

raw = pd.concat(all_gex, ignore_index=True)
print(f"Total rows across {NUM_EXPIRIES} expirations: {len(raw)}")

Total rows across 1 expirations: 213


In [13]:
# Cell 14 — compute GEX for every row
def compute_row_gex(row):
    K   = float(row["strike"])
    oi  = float(row["openInterest"])
    iv  = float(row["impliedVolatility"])
    T   = float(row["T"])
    g   = bs_gamma(spot, K, T, RATE, iv)
    # return row["sign"] * g * oi * 100 * spot**2 * 0.01
    return row["sign"] * g * oi * 100

raw["gex"] = raw.apply(compute_row_gex, axis=1)
print("GEX computed OK")

GEX computed OK


In [14]:
# Cell 15 — aggregate by strike, then slice to NUM_STRIKES centered on spot
gex_by_strike = (
    raw.groupby("strike")["gex"]
    .sum()
    .reset_index()
    .sort_values("strike")
    .reset_index(drop=True)
)

# find the index of the strike closest to spot
spot_idx = (gex_by_strike["strike"] - spot).abs().idxmin()

# take NUM_STRIKES/2 on each side
half      = NUM_STRIKES // 2
idx_min   = max(0, spot_idx - half)
idx_max   = min(len(gex_by_strike) - 1, spot_idx + half)
gex_plot  = gex_by_strike.iloc[idx_min:idx_max].reset_index(drop=True)

print(f"Strikes in range: {len(gex_plot)}  "
      f"(${gex_plot['strike'].min():.0f} — ${gex_plot['strike'].max():.0f})")
print(f"Net GEX: {fmt(gex_plot['gex'].sum())}")

Strikes in range: 50  ($704 — $775)
Net GEX: +168.3K


In [15]:
# Cell 16 — recompute levels from the aggregated data
strikes_arr = gex_plot["strike"].values
values_arr  = gex_plot["gex"].values
cumulative  = np.cumsum(values_arr)

gamma_flip = None
for i in range(1, len(cumulative)):
    if cumulative[i-1] * cumulative[i] <= 0:
        gamma_flip = strikes_arr[i]
        break

pos        = gex_plot[gex_plot["gex"] > 0]
neg        = gex_plot[gex_plot["gex"] < 0]
call_wall  = pos.loc[pos["gex"].idxmax(), "strike"] if len(pos) else None
put_wall   = neg.loc[neg["gex"].idxmin(), "strike"] if len(neg) else None
net_gex    = gex_plot["gex"].sum()
regime_str = "▲ positive gamma" if net_gex >= 0 else "▼ negative gamma"

print(f"Gamma flip: ${gamma_flip:.2f}" if gamma_flip else "Gamma flip: not found")
print(f"Call wall:  ${call_wall:.2f}"  if call_wall  else "Call wall:  not found")
print(f"Put wall:   ${put_wall:.2f}"   if put_wall   else "Put wall:   not found")

Gamma flip: $728.00
Call wall:  $730.00
Put wall:   $727.00


## Levels

In [16]:
# # Cell 11 — gamma flip (where cumulative GEX crosses zero)
# cumulative = gex_by_strike["gex"].cumsum().values
# strikes    = gex_by_strike["strike"].values

# gamma_flip = None
# for i in range(1, len(cumulative)):
#     if cumulative[i-1] * cumulative[i] <= 0:
#         gamma_flip = strikes[i]
#         break

# print(f"Gamma flip: ${gamma_flip:.2f}" if gamma_flip else "Gamma flip: not found in range")

In [17]:
# # Cell 12 — call wall and put wall
# pos = gex_by_strike[gex_by_strike["gex"] > 0]
# neg = gex_by_strike[gex_by_strike["gex"] < 0]

# call_wall = pos.loc[pos["gex"].idxmax(), "strike"] if len(pos) else None
# put_wall  = neg.loc[neg["gex"].idxmin(), "strike"] if len(neg) else None

# print(f"Call wall:  ${call_wall:.2f}" if call_wall else "Call wall:  not found")
# print(f"Put wall:   ${put_wall:.2f}"  if put_wall  else "Put wall:   not found")
# print(f"Spot:       ${spot:.2f}")

In [18]:
# # Cell 13 — net GEX and regime
# net_gex = gex_by_strike["gex"].sum()
# regime  = "POSITIVE gamma (dealers suppress moves)" if net_gex >= 0 else "NEGATIVE gamma (dealers amplify moves)"

# print(f"Net GEX: {net_gex:,.0f}")
# print(f"Regime:  {regime}")

## Plot

In [20]:
# Cell 17 — plot with plotly
import plotly.graph_objects as go

strikes = gex_by_strike["strike"].values
values  = gex_by_strike["gex"].values
colors  = ["#1D9E75" if v >= 0 else "#E24B4A" for v in values]

fig = go.Figure()

# ── bars ──────────────────────────────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=strikes,
    y=values,
    marker_color=colors,
    width=[0.7] * len(strikes),
    hovertemplate="Strike: $%{x}<br>GEX: %{customdata}<extra></extra>",
    customdata=[fmt(v) for v in values],
))

# ── key level vertical lines ──────────────────────────────────────────────────
y_min, y_max = values.min(), values.max()
padding      = (y_max - y_min) * 0.08

for price, color, label in [
    (gamma_flip, "#F0992B", "Gamma flip"),
    (call_wall,  "#1D9E75", "Call wall"),
    (put_wall,   "#E24B4A", "Put wall"),
    (spot,       "#7F77DD", "Spot"),
]:
    if price is None: continue

    fig.add_vline(
        x=price, line_color=color,
        line_dash="dash", line_width=1.5, opacity=0.9,
    )
    fig.add_annotation(
        x=price, y=y_max + padding,
        text=f"<b>{label}</b><br>${price:.0f}",
        showarrow=False,
        font=dict(color=color, size=11),
        align="left", xanchor="left",
    )

# ── zero line ─────────────────────────────────────────────────────────────────
fig.add_hline(y=0, line_color="#aaaaaa", line_width=1)

# ── layout ────────────────────────────────────────────────────────────────────
regime_str = "▲ positive gamma" if net_gex >= 0 else "▼ negative gamma"

# Cell — dark theme layout (replace previous update_layout)
fig.update_layout(
    title=dict(
        text=(
            f"QQQ GEX by Strike  ·  expiry {exp}  ·  net {fmt(net_gex)}  ·  {regime_str}<br>"
            f"<sup>spot ${spot:.2f}   call wall ${call_wall:.0f}   "
            f"put wall ${put_wall:.0f}   gamma flip ${gamma_flip:.0f}</sup>"
        ),
        font=dict(size=14, color="#e0e0e0"),
    ),
    xaxis=dict(
        title="Strike",
        title_font=dict(color="#999"),
        tickmode="linear",
        dtick=5,
        tickangle=-45,
        showgrid=False,
        tickfont=dict(color="#999"),
        linecolor="#444",
    ),
    yaxis=dict(
        title="GEX ($)",
        title_font=dict(color="#999"),
        tickformat=".3s",
        gridcolor="#2a2a2a",
        zerolinecolor="#555",
        tickfont=dict(color="#999"),
    ),
    plot_bgcolor="#0f0f0f",
    paper_bgcolor="#0f0f0f",
    showlegend=False,
    height=700,
    margin=dict(t=100, b=80, l=80, r=40),
    bargap=0.3,
)

fig.show()